# YouTube Shorts Virality — Feature Engineering

**Goal:** predict, from a Short's first few hours, whether it will go viral.

**Setup carried over from `EDA_final.ipynb`**
- **Outcome:** views at **72 hours**, using the snapshot nearest 72h within ±2h.
- **Viral:** 72h views in the **top 10%**. The threshold is computed on the **training split only**, so the test set never influences it.
- **Snapshot rules:** `snapshot_as_of` for features (no look-ahead past the cutoff), `snapshot_nearest` for outcomes.

**Population:** all videos with a 72h outcome, from **every country**. Each video keeps its channel's `channel_country`, so the same split can later be filtered to US-only videos.

Data comes from `shorts-data-final.zip`, read directly without unzipping.

In [ ]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", 50)

DATA_ZIP = Path("shorts-data-final.zip")
ZIP_DATA_DIR = "shorts-data/data/"

OUTCOME_HOURS = 72     # outcome = views at 72h
OUTCOME_TOL = 2        # nearest snapshot within ± this many hours
VIRAL_QUANTILE = 0.90  # viral = top 10% of training-set 72h views
EARLY_WINDOW = 1       # an "as of h" feature is the latest snapshot with age in [h - 1, h]

TEST_SIZE = 0.20
RANDOM_STATE = 42

## Load the data from the zip

In [ ]:
def to_utc(values):
    # The collector writes ISO-8601 timestamps both with and without fractional seconds.
    return pd.to_datetime(values, utc=True, format="ISO8601", errors="coerce")

with zipfile.ZipFile(DATA_ZIP) as z:
    videos = pd.read_csv(z.open(ZIP_DATA_DIR + "videos.csv"))
    channels = pd.read_csv(z.open(ZIP_DATA_DIR + "channels.csv"))
    countries = pd.read_csv(z.open(ZIP_DATA_DIR + "channel_countries.csv"))
    snapshot_files = sorted(n for n in z.namelist()
                            if n.startswith(ZIP_DATA_DIR + "observations/") and n.endswith(".csv"))
    observations = pd.concat((pd.read_csv(z.open(n)) for n in snapshot_files), ignore_index=True)

for col in ["first_seen_at", "published_at"]:
    videos[col] = to_utc(videos[col])
for col in ["first_seen_at", "channel_created_at"]:
    channels[col] = to_utc(channels[col])
countries["country_checked_at"] = to_utc(countries["country_checked_at"])
observations["observed_at"] = to_utc(observations["observed_at"])
for col in ["age_hours", "view_count", "like_count", "comment_count"]:
    observations[col] = pd.to_numeric(observations[col], errors="coerce")
observations.sort_values(["video_id", "observed_at"], inplace=True, ignore_index=True)

DATA_SNAPSHOT_AT = observations["observed_at"].max()
print(f"DATA SNAPSHOT: {DATA_SNAPSHOT_AT:%Y-%m-%d %H:%M} UTC")
print(f"videos:        {videos.shape}")
print(f"channels:      {channels.shape}")
print(f"countries:     {countries.shape}")
print(f"observations:  {observations.shape} from {len(snapshot_files)} snapshot files")

## Snapshot selection rules

Copied from `EDA_final.ipynb` so features and outcomes are selected exactly the same way.

- `snapshot_as_of(obs, h, window)`: the **latest** snapshot with age between `h - window` and `h`. Used for **features**, so nothing observed after the cutoff can leak in.
- `snapshot_nearest(obs, h, tol)`: the snapshot **closest** to age `h` within `± tol`. Used for **outcomes**.

In [ ]:
def snapshot_as_of(obs, hour, window, required=("view_count",)):
    candidates = obs[obs["age_hours"].between(hour - window, hour)].dropna(subset=list(required))
    return (candidates.sort_values(["video_id", "age_hours"])
            .drop_duplicates("video_id", keep="last")
            .set_index("video_id"))

def snapshot_nearest(obs, hour, tol, required=("view_count",)):
    candidates = obs[obs["age_hours"].between(hour - tol, hour + tol)].dropna(subset=list(required))
    candidates = candidates.assign(_distance=(candidates["age_hours"] - hour).abs())
    return (candidates.sort_values(["video_id", "_distance", "age_hours"])
            .drop_duplicates("video_id", keep="first")
            .drop(columns="_distance")
            .set_index("video_id"))

## Modeling population: every video with a 72h outcome

A video needs a 72h snapshot to get a label. Videos without one are excluded: mostly videos published too recently to reach 72h before the data snapshot, plus a few with collection gaps around 72h. Country is attached but **not** filtered on.

In [ ]:
TARGET = f"views_{OUTCOME_HOURS}h"

outcome = snapshot_nearest(observations, OUTCOME_HOURS, OUTCOME_TOL)["view_count"].rename(TARGET)

countries_latest = countries.sort_values("country_checked_at").drop_duplicates("channel_id", keep="last")
population = (
    videos.merge(countries_latest[["channel_id", "channel_country"]], on="channel_id", how="left", validate="many_to_one")
    .merge(outcome, left_on="video_id", right_index=True, how="inner")
    .reset_index(drop=True)
)

hours_old = (DATA_SNAPSHOT_AT - videos["published_at"]).dt.total_seconds() / 3600
excluded = videos[~videos["video_id"].isin(population["video_id"])]
too_recent = (hours_old[excluded.index] < OUTCOME_HOURS + OUTCOME_TOL).sum()
print(f"videos with a {OUTCOME_HOURS}h outcome: {len(population):,} of {len(videos):,}")
print(f"excluded: {len(excluded):,} ({too_recent:,} too recent to reach {OUTCOME_HOURS}h, "
      f"{len(excluded) - too_recent:,} old enough but no snapshot within ±{OUTCOME_TOL}h)")
print(f"channels: {population['channel_id'].nunique():,}")
print("\ntop channel countries (NaN = channel did not state one):")
print(population["channel_country"].value_counts(dropna=False).head(8).to_string())

## 80/20 train/test split

Two choices beyond a plain random split:

1. **Grouped by channel.** Every video from a channel goes to the same side. Some channels have dozens of videos in the data (one has over 100). If a channel appeared on both sides, the model could do well on the test set by recognizing the channel (same subscriber count, same channel age) rather than learning what makes videos go viral. Grouping makes the test set behave like **new creators**.
2. **Stratified by 72h views.** Videos are binned into deciles of 72h views, and each side gets a similar mix, so the test set gets its fair share of top-10% videos.

`StratifiedGroupKFold` with 5 folds does both: one fold (≈20%) becomes the test set, the other four (≈80%) the training set. Because channels vary in size, the split is close to 80/20 but not exact.

In [ ]:
view_decile = pd.qcut(population[TARGET].rank(method="first"), 10, labels=False)

splitter = StratifiedGroupKFold(n_splits=round(1 / TEST_SIZE), shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(population, view_decile, groups=population["channel_id"]))
population["split"] = "train"
population.loc[test_idx, "split"] = "test"

train_videos = population[population["split"] == "train"].reset_index(drop=True)
test_videos = population[population["split"] == "test"].reset_index(drop=True)

shared_channels = set(train_videos["channel_id"]) & set(test_videos["channel_id"])
assert not shared_channels, "a channel appears in both train and test"
print(f"train: {len(train_videos):,} videos ({len(train_videos) / len(population):.1%}), "
      f"{train_videos['channel_id'].nunique():,} channels")
print(f"test:  {len(test_videos):,} videos ({len(test_videos) / len(population):.1%}), "
      f"{test_videos['channel_id'].nunique():,} channels")
print("channels in both: 0")

## Viral label: threshold from the training set only

The 90th percentile of 72h views is computed on **train**, then applied to both sets. Test videos never influence the threshold.

In [ ]:
VIRAL_THRESHOLD = train_videos[TARGET].quantile(VIRAL_QUANTILE)
for frame in (population, train_videos, test_videos):
    frame["viral"] = frame[TARGET] >= VIRAL_THRESHOLD

print(f"viral threshold (train 90th percentile): {VIRAL_THRESHOLD:,.0f} views at {OUTCOME_HOURS}h")
summary = pd.DataFrame({
    "videos": population.groupby("split").size(),
    "viral": population.groupby("split")["viral"].sum(),
    "viral rate": population.groupby("split")["viral"].mean().round(3),
    f"median {TARGET}": population.groupby("split")[TARGET].median(),
    "US channels": population.groupby("split")["channel_country"].apply(lambda s: (s == "US").sum()),
}).loc[["train", "test"]]
summary

## Save the split

Saves which side each video is on, so teammates (and later notebooks) use the exact same split. To restrict to US videos later, filter both sides on `channel_country == "US"`. Because the split is by channel, the filtered sets still never share a channel.

In [ ]:
SPLIT_PATH = Path("results") / "train_test_split.csv"
SPLIT_PATH.parent.mkdir(exist_ok=True)
population[["video_id", "channel_id", "channel_country", "split"]].to_csv(SPLIT_PATH, index=False)
print(f"saved {len(population):,} rows to {SPLIT_PATH}")